In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),                                        # Pretvara sliku u PyTorch tenzor (vrednosti 0.0 - 1.0)
    transforms.Lambda(lambda img: torch.rot90(img, 1, [1, 2])), # Rotacija da slova ne budu "na bok"
    transforms.Lambda(lambda img: torch.flip(img, [2])),          # Horizontalno ogledalo ako je potrebno
    transforms.Normalize((0.5,), (0.5,))                          # Normalizacija na opseg [-1, 1]
])

train_dataset = datasets.EMNIST(
    root='./data', 
    split='byclass', 
    train=True, 
    download=True, 
    transform=transform
)
test_dataset = datasets.EMNIST(
    root='./data',
    split='byclass',
    train= False,
    download=True,
    transform=transform   
)

BATCH_SIZE=64

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
test_loader=DataLoader(test_dataset, batch_size=BATCH_SIZE,shuffle=True)

data_iter = iter(train_loader)
images, labels = next(data_iter)
print(f"Dimenzije batch-a slika: {images.shape}")  # Očekujemo [64, 1, 28, 28]
print(f"Dimenzije batch-a labela: {labels.shape}") # Očekujemo [64]

Dimenzije batch-a slika: torch.Size([64, 1, 28, 28])
Dimenzije batch-a labela: torch.Size([64])


In [2]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channles=1, patch_size=4, emb_dim=64, img_size=28): #64 filtera, in_ch je dimenzija boje
        super().__init__()
        self.patch_size = patch_size

        self.proj = nn.Conv2d( #izmerimo koliko se slazu filteri sa slikom
            in_channels=in_channles,
            out_channels=emb_dim,
            kernel_size=patch_size,
            stride=patch_size
        )
    def forwrad(self,x):
        x=self.proj(x)
        x=x.flatten(2) # pretvorimo u 'string', cuvamo prve dve dim
        x=x.transpose(1,2) # menjamo mesta 2 i 3 dim, da bi svaki od 49 patcheva dobio vektor od 64 dim// takav nam oblik treba za transformer

        return x


In [30]:
x=PatchEmbedding
x.forwrad(x,train_loader[1])

TypeError: 'DataLoader' object is not subscriptable